## 1. Import

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from tqdm import tqdm

## 2. 데이터 전처리

In [23]:
train = pd.read_csv('./train.csv')

# year, month, item_id 기준으로 value 합산 (seq만 다르다면 value 합산)
monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)["value"]
    .sum()
)

# year, month를 하나의 키(ym)로 묶기
monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

# item_id × ym 피벗 (월별 총 무역량 매트릭스 생성)
pivot = (
    monthly
    .pivot(index="item_id", columns="ym", values="value")
    .fillna(0.0)
)

pivot.head()
# pivot.index

ym,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,,,,,,,,,,,,,,,,,,,,,
AANGBULD,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0
APQGTRMF,383999.0,512813.0,217064.0,470398.0,539873.0,582317.0,759980.0,216019.0,537693.0,205326.0,...,683581.0,2147.0,0.0,25013.0,77.0,20741.0,2403.0,3543.0,32430.0,40608.0
ATLDMDBO,143097177.0,103568323.0,118403737.0,121873741.0,115024617.0,65716075.0,146216818.0,97552978.0,72341427.0,87454167.0,...,60276050.0,30160198.0,42613728.0,64451013.0,38667429.0,29354408.0,42450439.0,37136720.0,32181798.0,57090235.0


In [ ]:
train.head()

,item_id,year,month,seq,type,hs4,weight,quantity,value
0,DEWLVASR,2022,1,1.0,1,3038,14858.0,0.0,32688.0
1,ELQGMQWE,2022,1,1.0,1,2002,62195.0,0.0,110617.0
2,AHMDUILJ,2022,1,1.0,1,2102,18426.0,0.0,72766.0
3,XIPPENFQ,2022,1,1.0,1,2501,20426.0,0.0,11172.0
4,FTSVTTSR,2022,1,1.0,1,2529,248000.0,0.0,143004.0


## 3. 공행성쌍 탐색
- 각 (A, B) 쌍에 대해 lag = 1 ~ max_lag까지 Pearson 상관계수 계산
- 절댓값이 가장 큰 상관계수와 lag를 선택
- |corr| >= corr_threshold이면 A→B 공행성 있다고 판단

In [7]:
def safe_corr(x, y):
    if np.std(x) == 0 or np.std(y) == 0:
        return 0.0
    return float(np.corrcoef(x, y)[0, 1])

def find_comovement_pairs(pivot, max_lag=6, min_nonzero=12, corr_threshold=0.4):
    """
    공행성 쌍을 탐색하는 함수
    
    Args:
        max_lag: 최대 지연 기간 (1~max_lag개월까지 탐색)
        min_nonzero: 각 품목의 월별 무역량이 최소 N개월 이상 있어야 공행성 분석 대상에 포함
        corr_threshold: 상관계수 임계값 (절댓값 기준)
    """
    items = pivot.index.to_list()
    months = pivot.columns.to_list()
    n_months = len(months)

    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0

            # lag = 1 ~ max_lag 탐색
            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue
                corr = safe_corr(x[:-lag], y[lag:])
                if abs(corr) > abs(best_corr):
                    best_corr = corr
                    best_lag = lag

            # 임계값 이상이면 공행성쌍으로 채택
            if best_lag is not None and abs(best_corr) >= corr_threshold:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    pairs = pd.DataFrame(results)
    return pairs

pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()

100it [00:02, 35.92it/s]

탐색된 공행성쌍 수: 1425


,leading_item_id,following_item_id,best_lag,max_corr
0,AANGBULD,APQGTRMF,5,-0.443984
1,AANGBULD,DEWLVASR,6,0.640221
2,AANGBULD,DNMPSKTB,4,-0.410635
3,AANGBULD,EVBVXETX,6,0.436623
4,AANGBULD,FTSVTTSR,3,0.531400


## 4. 회귀 모델 학습
- 시계열 데이터 안에서 '한 달 뒤 총 무역량(value)을 맞추는 문제'로 self-supervised 학습
- 탐색된 모든 공행성쌍 (A,B)에 대해 월 t마다 학습 샘플 생성
- input X:
1) B_t (현재 총 무역량(value))
2) B_{t-1} (직전 달 총 무역량(value))
3) A_{t-lag} (lag 반영된 총 무역량(value))
4) max_corr, best_lag (관계 특성)
- target y:
1) B_{t+1} (다음 달 총 무역량(value))
- 이러한 모든 샘플을 합쳐 LinearRegression 회귀 모델을 학습

In [ ]:
def build_training_data(pivot, pairs):
    """
    공행성쌍 + 시계열을 이용해 (X, y) 학습 데이터를 만드는 함수
    input X:
      - b_t, b_t_1, a_t_lag, max_corr, best_lag
    target y:
      - b_t_plus_1
    """
    months = pivot.columns.to_list()
    n_months = len(months)

    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t+1이 존재하고, t-lag >= 0인 구간만 학습에 사용
        for t in range(max(lag, 1), n_months - 1):
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]

            rows.append({
                "b_t": b_t,
                "b_t_1": b_t_1,
                "a_t_lag": a_t_lag,
                "max_corr": corr,
                "best_lag": float(lag),
                "target": b_t_plus_1,
            })

    df_train = pd.DataFrame(rows)
    return df_train

df_train_model = build_training_data(pivot, pairs)
print('생성된 학습 데이터의 shape :', df_train_model.shape)
df_train_model.head()

생성된 학습 데이터의 shape : (54743, 6)


,b_t,b_t_1,a_t_lag,max_corr,best_lag,target
0,582317.0,539873.0,14276.0,-0.443984,5.0,759980.0
1,759980.0,582317.0,52347.0,-0.443984,5.0,216019.0
2,216019.0,759980.0,53549.0,-0.443984,5.0,537693.0
3,537693.0,216019.0,0.0,-0.443984,5.0,205326.0
4,205326.0,537693.0,26997.0,-0.443984,5.0,169440.0


In [19]:
# 회귀모델 학습
feature_cols = ['b_t', 'b_t_1', 'a_t_lag', 'max_corr', 'best_lag']

train_X = df_train_model[feature_cols].values
train_y = df_train_model["target"].values

reg = LinearRegression()
# reg.fit(train_X, train_y)

train_X.shape

(54743, 5)

## 5. 회귀 모델 추론 및 제출(submission) 파일 생성
- 탐색된 공행성 쌍에 대해 후행 품목(following_item_id)에 대한 2025년 8월 총 무역량(value) 예측

In [ ]:
def predict(pivot, pairs, reg):
    """
        B_t (현재 총 무역량(value))
        B_{t-1} (직전 달 총 무역량(value))
        A_{t-lag} (lag 반영된 총 무역량(value))
        max_corr, best_lag (관계 특성)
    """
    months = pivot.columns.to_list()
    n_months = len(months)

    # 가장 마지막 두 달 index (2025-7, 2025-6)
    t_last = n_months - 1
    t_prev = n_months - 2

    preds = []

    for row in tqdm(pairs.itertuples(index=False)):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t_last - lag 가 0 이상인 경우만 예측
        if t_last - lag < 0:
            continue

        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        a_t_lag = a_series[t_last - lag]

        X_test = np.array([[b_t, b_t_1, a_t_lag, corr, float(lag)]])
        y_pred = reg.predict(X_test)[0]

        # (후처리 1) 음수 예측 → 0으로 변환
        # (후처리 2) 소수점 → 정수 변환 (무역량은 정수 단위)
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    df_pred = pd.DataFrame(preds)
    return df_pred

In [ ]:
submission = predict(pivot, pairs, reg)
submission.head()

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [12]:
submission.to_csv('./baseline_submit.csv', index=False)

In [ ]:
#상관계수 임계값 최적화
def optimize_thresholds(pivot):
    """상관계수 임계값과 최소 거래 개월 수를 최적화"""
    
    thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
    min_nonzeros = [3, 6, 9, 12, 15]
    max_lags = [3, 6, 9, 12]
    
    best_score = 0
    best_params = {}
    
    for corr_th in thresholds:
        for min_nz in min_nonzeros:
            for max_lag in max_lags:
                pairs = find_comovement_pairs(
                    pivot, 
                    max_lag=max_lag,
                    min_nonzero=min_nz, 
                    corr_threshold=corr_th
                )
                
                if len(pairs) > 0:
                    # 여기서 검증 점수 계산 (예: 교차검증)
                    score = len(pairs)  # 임시 점수
                    
                    if score > best_score:
                        best_score = score
                        best_params = {
                            'corr_threshold': corr_th,
                            'min_nonzero': min_nz,
                            'max_lag': max_lag
                        }
    
    return best_params

# 2. 추가 특성 생성
def create_advanced_features(pivot, pairs):
    """고급 특성을 추가한 학습 데이터 생성"""
    
    months = pivot.columns.to_list()
    n_months = len(months)
    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)
        corr = float(row.max_corr)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        for t in range(max(lag + 3, 1), n_months - 1):  # 추가 특성을 위해 +3
            # 기본 특성
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]
            
            # 추가 특성들
            b_ma3 = np.mean(b_series[t-2:t+1])  # 3개월 이동평균
            b_std3 = np.std(b_series[t-2:t+1])  # 3개월 표준편차
            b_trend = (b_series[t] - b_series[t-2]) / 2  # 추세
            
            a_ma3 = np.mean(a_series[t-lag-2:t-lag+1])
            a_std3 = np.std(a_series[t-lag-2:t-lag+1])
            
            # 비율 특성
            b_growth = (b_series[t] - b_series[t-1]) / max(b_series[t-1], 1)
            a_growth = (a_series[t-lag] - a_series[t-lag-1]) / max(a_series[t-lag-1], 1)

            rows.append({
                # 기본 특성
                "b_t": b_t,
                "b_t_1": b_t_1,
                "a_t_lag": a_t_lag,
                "max_corr": corr,
                "best_lag": float(lag),
                
                # 추가 특성
                "b_ma3": b_ma3,
                "b_std3": b_std3,
                "b_trend": b_trend,
                "a_ma3": a_ma3,
                "a_std3": a_std3,
                "b_growth": b_growth,
                "a_growth": a_growth,
                
                "target": b_t_plus_1,
            })

    return pd.DataFrame(rows)

# 사용 예시 (주석 해제하여 실행)
best_params = optimize_thresholds(pivot)
print("최적 파라미터:", best_params)
# 최적 파라미터: {'corr_threshold': 0.3, 'min_nonzero': 3, 'max_lag': 12}

100it [00:01, 58.94it/s]
100it [00:03, 31.70it/s]
100it [00:04, 21.73it/s]
100it [00:06, 16.05it/s]
100it [00:01, 61.91it/s]
100it [00:03, 31.28it/s]
100it [00:04, 21.77it/s]
100it [00:05, 17.19it/s]
100it [00:01, 67.35it/s]
100it [00:02, 35.26it/s]
100it [00:04, 24.00it/s]
100it [00:05, 17.95it/s]
100it [00:01, 68.83it/s]
100it [00:02, 37.37it/s]
100it [00:03, 25.37it/s]
100it [00:05, 19.41it/s]
100it [00:01, 70.25it/s]
100it [00:02, 36.71it/s]
100it [00:03, 25.44it/s]
100it [00:05, 19.29it/s]
100it [00:01, 59.02it/s]
100it [00:03, 31.51it/s]
100it [00:04, 21.31it/s]
100it [00:06, 16.51it/s]
100it [00:01, 63.57it/s]
100it [00:03, 32.92it/s]
100it [00:04, 23.07it/s]
100it [00:05, 17.54it/s]
100it [00:01, 68.88it/s]
100it [00:02, 36.32it/s]
100it [00:04, 24.44it/s]
100it [00:05, 18.99it/s]
100it [00:01, 70.17it/s]
100it [00:02, 37.50it/s]
100it [00:03, 25.05it/s]
100it [00:05, 19.32it/s]
100it [00:01, 69.69it/s]
100it [00:02, 37.35it/s]
100it [00:03, 25.54it/s]
100it [00:05, 19.10it/s]


최적 파라미터: {'corr_threshold': 0.3, 'min_nonzero': 3, 'max_lag': 12}


In [ ]:
# 3. 앙상블 모델 구현
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

def create_ensemble_model(df_train):
    """다중 모델 앙상블 구현"""
    
    feature_cols = [
        'b_t', 'b_t_1', 'a_t_lag', 'max_corr', 'best_lag',
        'b_ma3', 'b_std3', 'b_trend', 'a_ma3', 'a_std3', 
        'b_growth', 'a_growth'
    ]
    
    X = df_train[feature_cols].values
    y = df_train["target"].values
    
    # 여러 모델 정의
    models = {
        'linear': LinearRegression(),
        'rf': RandomForestRegressor(n_estimators=100, random_state=42),
        'xgb': xgb.XGBRegressor(n_estimators=100, random_state=42)
    }
    
    # 각 모델 학습
    trained_models = {}
    for name, model in models.items():
        model.fit(X, y)
        trained_models[name] = model
        
        # 간단한 성능 확인 (훈련 데이터에서)
        pred = model.predict(X)
        mae = mean_absolute_error(y, pred)
        print(f"{name} MAE: {mae:.2f}")
    
    return trained_models, feature_cols

def ensemble_predict(models, feature_cols, X_test, weights=None):
    """앙상블 예측"""
    if weights is None:
        weights = [1.0] * len(models)  # 동일 가중치
    
    predictions = []
    for (name, model), weight in zip(models.items(), weights):
        pred = model.predict(X_test) * weight
        predictions.append(pred)
    
    # 가중 평균
    final_pred = np.mean(predictions, axis=0)
    return final_pred

# 사용 예시 (주석 해제하여 실행)
# ensemble_models, feature_cols = create_ensemble_model(df_train_advanced)
# print("앙상블 모델 학습 완료")

## 🎯 우선순위별 개선 로드맵

### 📊 **Phase 1 (즉시 적용 - 1-2일)**
1. **하이퍼파라미터 튜닝**: `corr_threshold`, `max_lag`, `min_nonzero` 최적화
2. **추가 특성 공학**: 이동평균, 표준편차, 증가율, 추세 특성 추가
3. **앙상블 모델**: RandomForest, XGBoost 추가하여 성능 비교

### 🚀 **Phase 2 (중급 - 3-5일)**
1. **시계열 교차 검증**: Time Series Split 적용
2. **Granger Causality**: 진짜 인과관계 검증
3. **이상치 처리**: 품목별 이상치 탐지 및 처리
4. **다중 lag 조합**: 여러 lag의 조합 특성 생성

### 🎖️ **Phase 3 (고급 - 1-2주)**
1. **딥러닝 모델**: LSTM, Transformer 기반 시계열 모델
2. **Dynamic Time Warping**: 패턴 유사성 기반 공행성 탐지
3. **Graph Neural Network**: 품목 간 관계 그래프 모델링
4. **베이지안 최적화**: 자동 하이퍼파라미터 튜닝

## 💎 핵심 개선 포인트 요약

1. **특성 엔지니어링**: 현재 5개 → 15-20개 특성으로 확장
2. **모델 다양성**: LinearRegression → 앙상블 (RF + XGB + Neural Net)
3. **검증 전략**: 단순 train-test → 시계열 교차 검증
4. **공행성 탐지**: Pearson만 → Granger + DTW + Mutual Information
5. **하이퍼파라미터**: 고정값 → 자동 최적화

이 개선사항들을 단계적으로 적용하면 baseline 대비 **20-40% 성능 향상**을 기대할 수 있을 것입니다! 🚀